# 03 - 推理与测试

## 任务
- **方案1**：对比 Standard / Zero-shot COT / Few-shot COT 的效果
- **所有模型**：Baseline / SFT / DPO / GRPO 统一推理
- **生成提交文件**：输出 CSV 提交结果

## 运行环境
- CPU: 可跑通流程，速度慢
- GPU: 批量推理，速度快（推荐）

In [ ]:
import os, sys

if os.path.exists('/kaggle'):
    WORK_DIR = '/kaggle/working'
elif os.path.exists('/mnt/workspace'):
    WORK_DIR = '/mnt/workspace'
else:
    WORK_DIR = os.getcwd()
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

print(f"工作目录: {os.getcwd()}")

In [ ]:
import subprocess, sys
for p in ['transformers', 'peft', 'tqdm']:
    try: __import__(p)
    except: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])
print('依赖就绪')

In [ ]:
from utils.pipeline_config import get_paths, get_device

paths = get_paths()
device = get_device()
BATCH_SIZE = 16 if device != 'cpu' else 1

print(f"设备: {device}, 批大小: {BATCH_SIZE}")
print(f"模型: {paths['base_model']}")
print(f"数据: {paths['data_dir']}")
print(f"输出: {paths['output_dir']}")

import os
MODEL_PATHS = {
    'base': paths['base_model'],
    'scheme2_sft': os.path.join(paths['output_dir'], 'scheme2_cot', 'final'),
    'scheme3_dpo': os.path.join(paths['output_dir'], 'scheme3_dpo', 'final'),
    'scheme4_grpo': os.path.join(paths['output_dir'], 'scheme4_grpo', 'final'),
}

for name, path in MODEL_PATHS.items():
    exists = '✓' if os.path.exists(path) else '✗'
    print(f"  {exists} {name}: {path}")

## ===== 方案1：COT Prompt对比测试 =====

对比三种推理方式的效果差异

In [ ]:
from scheme1_cot.inference import load_model, inference_with_cot
from scheme1_cot.cot_prompts import get_cot_prompt
from utils.common import load_json

print('推理模块导入成功')

In [ ]:
test_data = load_json(os.path.join(paths['data_dir'], 'test.json'))
print(f"测试样本: {len(test_data)}条")

for pt in ['standard', 'zero_shot_simple', 'zero_shot', 'few_shot']:
    prompt = get_cot_prompt(pt)
    print(f"  [{pt}] {len(prompt)}字符")

In [ ]:
INFER_MODEL = paths['base_model']
INFER_PEFT = None

for name in ['scheme4_grpo', 'scheme3_dpo', 'scheme2_sft']:
    path = MODEL_PATHS.get(name)
    if path and os.path.exists(path):
        INFER_PEFT = path
        print(f"使用模型: {name} ({path})")
        break

if INFER_PEFT is None:
    print(f"使用基础模型（无微调）")

In [ ]:
print(f"加载模型: {INFER_MODEL}")
if INFER_PEFT:
    print(f"加载权重: {INFER_PEFT}")

model, tokenizer = load_model(INFER_MODEL, INFER_PEFT, device=device)
print('模型加载完成')

In [ ]:
TRAIN_PATH = os.path.join(paths['data_dir'], 'train.json')

if os.path.exists(TRAIN_PATH):
    train_for_eval = load_json(TRAIN_PATH)[:200]
    print(f'使用 train.json 子集做离线验证: {len(train_for_eval)}条')
    print()
    
    offline_results = {}
    for pt in ['standard', 'zero_shot', 'few_shot']:
        print(f'测试 {pt}...')
        preds = inference_with_cot(model, tokenizer, train_for_eval,
                                   prompt_type=pt,
                                   batch_size=BATCH_SIZE,
                                   device=device)
        correct = sum(1 for (qid, pred), item in zip(preds, train_for_eval)
                     if pred.strip() == str(item['answer']).strip())
        acc = correct / len(preds) if preds else 0
        offline_results[pt] = acc
        print(f'  {pt:15s}: {acc:.2%} ({correct}/{len(preds)})')
    
    print()
    best_prompt = max(offline_results, key=offline_results.get)
    print(f'✓ 最佳Prompt（离线验证）: {best_prompt} ({offline_results[best_prompt]:.2%})')
else:
    print('train.json 不存在，跳过离线验证')
    best_prompt = 'zero_shot'

In [ ]:
print('='*50)
print('方案1 Prompt对比 - 输出示例（前3条）')
print('='*50)

TEST_SIZE = 10
test_subset = test_data[:TEST_SIZE]
PROMPT_TYPES = ['standard', 'zero_shot', 'few_shot']

cot_results = {}
for pt in PROMPT_TYPES:
    print(f"\n--- [{pt}] ---")
    results = inference_with_cot(model, tokenizer, test_subset, 
                                  prompt_type=pt, 
                                  batch_size=BATCH_SIZE,
                                  device=device)
    cot_results[pt] = results
    for qid, pred in results[:3]:
        item = next(i for i in test_subset if i['id'] == qid)
        print(f"  ID {qid}: {pred}")
        print(f"  问题: {item['question'][:50]}...")

## ===== 生成提交文件 =====

In [ ]:
print(f"使用Prompt: {best_prompt}")
print(f"测试数据: {len(test_data)}条")

final_results = inference_with_cot(model, tokenizer, test_data,
                                  prompt_type=best_prompt,
                                  batch_size=BATCH_SIZE,
                                  device=device)
print(f"推理完成: {len(final_results)}条")

In [ ]:
from utils.common import save_csv

SUBMIT_PATH = os.path.join(paths['output_dir'], 'submit.csv')
os.makedirs(paths['output_dir'], exist_ok=True)
save_csv(final_results, SUBMIT_PATH)

print(f"提交文件已保存: {SUBMIT_PATH}")

print('\n结果预览（前10条）:')
for i, (qid, ans) in enumerate(final_results[:10]):
    print(f"  {qid}: {ans}")

In [ ]:
print('\n' + '='*50)
print('推理与测试完成')
print('='*50)
print(f"提交文件: {SUBMIT_PATH}")
print(f"样本总数: {len(final_results)}")
print(f"使用Prompt: {best_prompt}")
print(f"使用模型: {'微调模型' if INFER_PEFT else '基础模型'}")